<a href="https://colab.research.google.com/github/youmnaesam/DataScienceProjects/blob/main/Amazon_Fine_Food_Reviews_big_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install kaggle

In [6]:
!ls

sample_data


In [7]:
!ls /content/drive

ls: cannot access '/content/drive': No such file or directory


In [8]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [9]:
!kaggle datasets download -d snap/amazon-fine-food-reviews
!unzip amazon-fine-food-reviews.zip

Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0
100% 242M/242M [00:03<00:00, 71.2MB/s]

Archive:  amazon-fine-food-reviews.zip
  inflating: Reviews.csv             
  inflating: database.sqlite         
  inflating: hashes.txt              


In [ ]:
!mv Reviews.csv "/content/drive/My Drive/"

In [ ]:
#!mv Reviews.csv /content/drive/MyDrive/

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Reviews.csv')

Mounted at /content/drive


In [11]:
df = pd.read_csv('/content/drive/MyDrive/Reviews.csv')
df.to_parquet('/content/drive/MyDrive/Reviews.parquet')

In [14]:
df

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...
...,...,...,...,...,...,...,...,...,...,...
568449,568450,B001EO7N10,A28KG5XORO54AY,Lettie D. Carter,0,0,5,1299628800,Will not do without,Great for sesame chicken..this is a good if no...
568450,568451,B003S1WTCU,A3I8AFVPEE8KI5,R. Sawyer,0,0,2,1331251200,disappointed,I'm disappointed with the flavor. The chocolat...
568451,568452,B004I613EE,A121AA1GQV751Z,"pksd ""pk_007""",2,2,5,1329782400,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o..."
568452,568453,B004I613EE,A3IBEVCTXKNOH,"Kathy A. Welch ""katwel""",1,1,5,1331596800,Favorite Training and reward treat,These are the BEST treats for training and rew...


In [13]:
#df = pd.read_csv('/kaggle/input/amazon-fine-food-reviews/Reviews.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/amazon-fine-food-reviews/Reviews.csv'

In [ ]:
df

In [ ]:
"""
Project idea : Text Summarization using Machine Learning
"""

In [15]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.2 MB/s eta 0:00:00


In [16]:
import contractions
import pandas as pd
from tqdm.notebook import tqdm

tqdm.pandas()  # enables progress_apply for a progress bar

# Apply contraction expansion to each row
df['Summary'] = df['Summary'].progress_apply(lambda x: contractions.fix(str(x)))

  0%|          | 0/568454 [00:00<?, ?it/s]

In [17]:
df['Summary_clean'] = (
    df['Summary']
    .str.replace('!', '', regex=False)
    .str.lower()
)
df['Summary_clean'] = df['Summary_clean'].progress_apply(lambda x: contractions.fix(str(x)))

  0%|          | 0/568454 [00:00<?, ?it/s]

In [18]:
fillers = r'\b(very|so|absolutely|pretty|really)\b'

df['Summary_clean'] = (
    df['Summary_clean']
    .str.replace(fillers, '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

In [19]:
# 1️⃣ Count occurrences of each summary
summary_counts = df['Summary_clean'].value_counts()

# 2️⃣ Select the rare ones (last 60)
rare_summaries = summary_counts.tail(60).index.tolist()

# 3️⃣ Filter the original DataFrame for these rare summaries and keep 'Summary_clean' and 'Score'
rare_df = df[df['Summary_clean'].isin(rare_summaries)][['Summary_clean', 'Score']].copy()

print(rare_df)

                                            Summary_clean  Score
190297            tastes like the original - but low carb      5
190311                   amazing great for low carb diets      5
190313                           heinz sugar free ketchup      5
190314  love it, especially when on atkins or any low ...      5
190315  you can tell by the taste that it has sweetene...      2
190317           great stuff - my 7 year old son loves it      5
190318     taste exactly like heinz regular ketchup to me      5
190319                                      thanks, heinz      5
190364                           yummy, and without guilt      5
190367                          heaven in a little pastry      5
190369                             a top of the line food      5
190370  excellent salmon and sweet potato dog food; my...      5
190371  salmon dog food with fruit and vegetables; hea...      5
190377                       great deal, and tastes great      5
190380                   

In [20]:
print(df.groupby('Score')['Summary_clean'].nunique())

Score
1     26715
2     16814
3     25065
4     43980
5    158957
Name: Summary_clean, dtype: int64


In [21]:
# Step 1: summaries with consistent score (all same score)
score_consistency = df.groupby('Summary_clean')['Score'].nunique()
consistent        = score_consistency[score_consistency == 1].index
df_c              = df[df['Summary_clean'].isin(consistent)].copy()

# Step 2: summaries that appear more than once
counts    = df_c['Summary_clean'].value_counts()
frequent  = counts[counts > 1].index
df_c      = df_c[df_c['Summary_clean'].isin(frequent)].copy()

print(f"Unique summaries : {df_c['Summary_clean'].nunique():,}")
print(f"Shape            : {df_c.shape}")
print(f"\nTop 20:")
print(df_c[['Summary_clean','Score']].drop_duplicates().value_counts().head(20))

Unique summaries : 45,465
Shape            : (189144, 11)

Top 20:
Summary_clean                                                  Score
~~perfect on-the-go snack~~                                    5        1
" a miracle supplement"                                        5        1
"ach deliver me from ll bean."                                 5        1
"al dente" oatmeal                                             5        1
"amazing" said the 5 month old critic                          5        1
"arnold palmer" k-cup                                          5        1
"awesome " said my daughter-in-law                             5        1
"best canned cat food", says cat. :)                           5        1
"best if used by" date                                         3        1
zukes hipaction for dogs - works                               5        1
zukes hip action treats                                        5        1
zukes hip action                                  

In [22]:
print(f"Before: {df['Summary_clean'].nunique():,} unique summaries")
print(f"After : {df_c['Summary_clean'].nunique():,} unique summaries")
print(f"Rows  : {df_c.shape[0]:,}")

Before: 259,960 unique summaries
After : 45,465 unique summaries
Rows  : 189,144


In [23]:
# Step 3: Keep only short summaries (1-4 words) — more general and reusable
df_c['summary_word_count'] = df_c['Summary_clean'].str.split().str.len()

df_short = df_c[df_c['summary_word_count'] <= 4].copy()

# Step 4: Keep only summaries appearing more than 5 times
counts      = df_short['Summary_clean'].value_counts()
frequent    = counts[counts > 5].index
df_short    = df_short[df_short['Summary_clean'].isin(frequent)].copy()

print(f"Unique summaries : {df_short['Summary_clean'].nunique():,}")
print(f"Shape            : {df_short.shape}")
print(f"\nTop 20 summaries with scores:")
print(df_short[['Summary_clean', 'Score']].drop_duplicates()
      .merge(df_short['Summary_clean'].value_counts().rename('count'),
             left_on='Summary_clean', right_index=True)
      .sort_values('count', ascending=False)
      .head(20))

Unique summaries : 4,968
Shape            : (49720, 12)

Top 20 summaries with scores:
                 Summary_clean  Score  count
222            my favorite tea      5    211
463           love these chips      5    126
37491          love these bars      5    104
428            best chips ever      5    101
2721         best popcorn ever      5     85
2703             the best ever      5     83
3210       great hot chocolate      5     74
15432   cannot live without it      5     74
465          these are awesome      5     68
3654                  best tea      5     65
839               the best tea      5     64
491        the best chips ever      5     63
16470        best cookies ever      5     63
2817          best cereal ever      5     63
11822               perfection      5     61
10571              amazing tea      5     59
4240                we love it      5     57
18250          wonderful stuff      5     54
4831             best dog food      5     53
12160  my dog

In [24]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [25]:
from nltk.corpus import stopwords

STOP_WORDS = set(stopwords.words('english'))

def strip_leading_trailing_stopwords(text):
    words = text.split()
    # Remove leading stopwords
    while words and words[0] in STOP_WORDS:
        words.pop(0)
    # Remove trailing stopwords
    while words and words[-1] in STOP_WORDS:
        words.pop(-1)
    return ' '.join(words)

df_short['Summary_clean'] = df_short['Summary_clean'].apply(strip_leading_trailing_stopwords)

# Re-check after stripping
print(f"Unique summaries before: 5,064")
print(f"Unique summaries after : {df_short['Summary_clean'].nunique():,}")

# Show merged examples
print("\nTop 20 after merging:")
print(df_short['Summary_clean'].value_counts().head(20))

Unique summaries before: 5,064
Unique summaries after : 4,786

Top 20 after merging:
Summary_clean
best                   241
favorite tea           220
love                   176
best chips ever        164
best tea               129
love these bars        127
love these chips       126
best popcorn ever       97
cannot live without     94
awesome                 92
best cookies ever       84
best ever               83
great hot chocolate     74
best dog food           72
cats love               69
best cereal ever        69
great                   68
                        64
hated                   63
cat's favorite          62
Name: count, dtype: int64


In [26]:
# Use df_short as the training data
df_model = df_short.copy()

print(f"Shape            : {df_model.shape}")
print(f"Unique summaries : {df_model['Summary_clean'].nunique():,}")
print(f"\nScore distribution:")
print(df_model['Score'].value_counts().sort_index())
print(f"\nSample:")
print(df_model[['Summary_clean', 'Score']].sample(10, random_state=42))

Shape            : (49720, 12)
Unique summaries : 4,786

Score distribution:
Score
1     3839
2     1258
3     2181
4     3898
5    38544
Name: count, dtype: int64

Sample:
                    Summary_clean  Score
247077     best for dental health      5
463362             love love love      5
562220            perfect balance      4
24121       popchips are the best      5
553923  fanstastico bravo awesome      5
195706                 divine tea      5
95895               love my bears      5
132692             best chai ever      5
4008       bad flavor combination      2
511754                  mom loves      5


In [27]:
# Split by frequency
summary_counts  = df_model['Summary_clean'].value_counts()
train_summaries = summary_counts[summary_counts > 10].index
test_summaries  = summary_counts[summary_counts <= 10].index

df_train = df_model[df_model['Summary_clean'].isin(train_summaries)].copy()
df_test  = df_model[df_model['Summary_clean'].isin(test_summaries)].copy()

print(f"Train shape           : {df_train.shape}")
print(f"Test  shape           : {df_test.shape}")
print(f"Train unique summaries: {df_train['Summary_clean'].nunique():,}")
print(f"Test  unique summaries: {df_test['Summary_clean'].nunique():,}")

Train shape           : (24993, 12)
Test  shape           : (24727, 12)
Train unique summaries: 1,414
Test  unique summaries: 3,372


In [28]:
from sklearn.model_selection import train_test_split

# ── Split 80/20 within each summary group ──
df_train_list = []
df_test_list  = []

for summary, group in df_model.groupby('Summary_clean'):
    if len(group) == 1:
        df_train_list.append(group)  # only 1 row → goes to train
    else:
        tr, te = train_test_split(group, test_size=0.2, random_state=42)
        df_train_list.append(tr)
        df_test_list.append(te)

df_train = pd.concat(df_train_list, ignore_index=True)
df_test  = pd.concat(df_test_list,  ignore_index=True)

print(f"Train shape           : {df_train.shape}")
print(f"Test  shape           : {df_test.shape}")
print(f"Train unique summaries: {df_train['Summary_clean'].nunique():,}")
print(f"Test  unique summaries: {df_test['Summary_clean'].nunique():,}")
print(f"\nScore distribution in train:")
print(df_train['Score'].value_counts().sort_index())

Train shape           : (37319, 12)
Test  shape           : (12401, 12)
Train unique summaries: 4,786
Test  unique summaries: 4,786

Score distribution in train:
Score
1     2862
2      926
3     1617
4     2894
5    29020
Name: count, dtype: int64


In [29]:
# Balance the dataset by undersampling score 5
min_count = df_short['Score'].value_counts().drop(5).min()  # min count excluding score 5
print(f"Min count (excluding score 5): {min_count:,}")

# Sample each score to be at most 2x the min count
TARGET_PER_SCORE = min_count * 2

df_balanced = df_short.groupby('Score', group_keys=False).apply(
    lambda x: x.sample(min(len(x), TARGET_PER_SCORE), random_state=42)
)

print(f"\nShape after balancing: {df_balanced.shape}")
print(f"\nScore distribution after balancing:")
print(df_balanced['Score'].value_counts().sort_index())
print(f"\nUnique summaries: {df_balanced['Summary_clean'].nunique():,}")

Min count (excluding score 5): 1,258

Shape after balancing: (10987, 12)

Score distribution after balancing:
Score
1    2516
2    1258
3    2181
4    2516
5    2516
Name: count, dtype: int64

Unique summaries: 2,867


/tmp/ipykernel_7868/3556339777.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df_short.groupby('Score', group_keys=False).apply(


In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

# ── Split 80/20 ──
X_train, X_test, y_train, y_test = train_test_split(
    df_balanced['Summary_clean'],
    df_balanced['Score'],
    test_size=0.2,
    random_state=42,
    stratify=df_balanced['Score']
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"\nTrain score distribution:\n{y_train.value_counts().sort_index()}")
print(f"\nTest score distribution:\n{y_test.value_counts().sort_index()}")

Train: 8,789 | Test: 2,198

Train score distribution:
Score
1    2013
2    1006
3    1745
4    2013
5    2012
Name: count, dtype: int64

Test score distribution:
Score
1    503
2    252
3    436
4    503
5    504
Name: count, dtype: int64


In [31]:
# ── TF-IDF ──
tfidf   = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

# ── Train Models ──
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    print(f"\n⏳ Training {name}...")
    model.fit(X_train_tfidf, y_train)
    y_pred   = model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = {'model': model, 'accuracy': accuracy, 'y_pred': y_pred}
    print(f"✅ {name} Accuracy: {accuracy:.4f}")

# ── Best model ──
best_name  = max(results, key=lambda x: results[x]['accuracy'])
best_model = results[best_name]['model']
print(f"\n🏆 Best model: {best_name} ({results[best_name]['accuracy']:.4f})")

# ── Classification Report ──
print("\n📋 Classification Report:")
print(classification_report(y_test, results[best_name]['y_pred']))


⏳ Training Logistic Regression...
✅ Logistic Regression Accuracy: 0.9199

⏳ Training Random Forest...
✅ Random Forest Accuracy: 0.9800

🏆 Best model: Random Forest (0.9800)

📋 Classification Report:
              precision    recall  f1-score   support

           1       0.99      0.98      0.99       503
           2       0.99      1.00      0.99       252
           3       0.98      1.00      0.99       436
           4       0.98      0.97      0.98       503
           5       0.96      0.96      0.96       504

    accuracy                           0.98      2198
   macro avg       0.98      0.98      0.98      2198
weighted avg       0.98      0.98      0.98      2198



In [32]:
# ── Prediction Function ──
def predict_score(summary):
    cleaned  = summary.lower().strip()
    features = tfidf.transform([cleaned])
    pred     = best_model.predict(features)[0]
    proba    = best_model.predict_proba(features)[0]
    top3_idx = proba.argsort()[::-1][:3]

    print(f"📝 Summary : {summary}")
    print(f"✅ Score   : {pred}")
    print(f"\nTop 3 predictions:")
    for i in top3_idx:
        print(f"  Score {best_model.classes_[i]} → {proba[i]*100:.1f}%")
    return pred

# ── Test ──
predict_score("best chips ever")
predict_score("terrible product")
predict_score("love these bars")

📝 Summary : best chips ever
✅ Score   : 5

Top 3 predictions:
  Score 5 → 100.0%
  Score 4 → 0.0%
  Score 3 → 0.0%
📝 Summary : terrible product
✅ Score   : 1

Top 3 predictions:
  Score 1 → 100.0%
  Score 5 → 0.0%
  Score 4 → 0.0%
📝 Summary : love these bars
✅ Score   : 5

Top 3 predictions:
  Score 5 → 100.0%
  Score 4 → 0.0%
  Score 3 → 0.0%


np.int64(5)

In [33]:
print(f"df shape        : {df.shape}")
print(f"df_balanced shape: {df_balanced.shape}")
print(f"\ndf columns      : {df.columns.tolist()}")
print(f"\nSample:")
print(df_balanced[['Text', 'Summary_clean', 'Score']].head(5))

df shape        : (568454, 11)
df_balanced shape: (10987, 12)

df columns      : ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text', 'Summary_clean']

Sample:
                                                     Text  \
123053  This smells more like cat food. My dogs ate th...   
567815  This has to be without a doubt the WORST tasti...   
474056  I bought one of the Stash teas-don't remember ...   
402692  I have fed BB for 3yrs and right now I have my...   
295705  The Granola bars, with about a month left befo...   

                   Summary_clean  Score  
123053            salmon formula      1  
567815  worst hot chocolate ever      1  
474056             gluten in tea      1  
402692                  research      1  
295705        stale granola bars      1  


In [34]:
import re
import pandas as pd
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import nltk
nltk.download('stopwords', quiet=True)

STOP_WORDS = set(stopwords.words('english'))

# ════════════════════════════════════════════════════════
# CELL 1: Clean Text
# ════════════════════════════════════════════════════════
def collapse_repeated_chars(text):
    return re.sub(r'(.)\1{2,}', r'\1\1', str(text))

def strip_leading_trailing_stopwords(text):
    words = text.split()
    while words and words[0] in STOP_WORDS:
        words.pop(0)
    while words and words[-1] in STOP_WORDS:
        words.pop(-1)
    return ' '.join(words)

def clean_text(text):
    text = str(text).lower()
    text = collapse_repeated_chars(text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(w for w in text.split() if w not in STOP_WORDS)

def clean_summary(text):
    text = str(text).lower()
    text = collapse_repeated_chars(text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return strip_leading_trailing_stopwords(text)

df['Text_clean']    = df['Text'].apply(clean_text)
df['Summary_clean'] = df['Summary'].apply(clean_summary)
print(f"✅ Cleaning done: {df.shape}")

✅ Cleaning done: (568454, 12)


In [63]:
# CELL 2: Filter & Balance

df_c = df.copy()

print(f"✅ Dataset shape: {df_c.shape}")
print(f"✅ Unique summaries: {df_c['Summary_clean'].nunique():,}")
print(df_c['Score'].value_counts().sort_index())

✅ Dataset shape: (568454, 12)
✅ Unique summaries: 237,327
Score
1     52268
2     29769
3     42640
4     80655
5    363122
Name: count, dtype: int64


In [64]:
# ── Keep only unique Text_clean + Summary_clean combinations ──
df_c = df_c.drop_duplicates(subset=['Text_clean', 'Summary_clean']).copy()

print(f"✅ df_c shape after dedup: {df_c.shape}")
print(f"Unique summaries: {df_c['Summary_clean'].nunique():,}")
print(df_c['Score'].value_counts().sort_index())

✅ df_c shape after dedup: (394399, 12)
Unique summaries: 237,327
Score
1     36385
2     20820
3     29796
4     56127
5    251271
Name: count, dtype: int64


In [65]:
df_balanced

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,Summary_clean,summary_word_count
123053,123054,B000EGZ2R6,A3OIM2L0NQH5AT,Dawn,1,1,1,1335312000,Salmon Formula,This smells more like cat food. My dogs ate th...,salmon formula,2
567815,567816,B005K4Q68Q,A3FJ7K6XRP3MPM,J. Voss,1,2,1,1324166400,Worst Hot chocolate ever!,This has to be without a doubt the WORST tasti...,worst hot chocolate ever,4
474056,474057,B000CQC08C,A3OPH5QDQRBNPT,Marilyn49,3,11,1,1250985600,Gluten in Tea!!!,I bought one of the Stash teas-don't remember ...,gluten in tea,3
402692,402693,B003P9XFVO,AED380YNNHAO8,Vicki Quattlebaum,6,10,1,1313712000,Do your research,I have fed BB for 3yrs and right now I have my...,research,3
295705,295706,B003WLC4VC,A33QVK92E9R28T,"Jerry D. Goodwin ""Accountant""",1,1,1,1333152000,Stale Granola Bars,"The Granola bars, with about a month left befo...",stale granola bars,3
...,...,...,...,...,...,...,...,...,...,...,...,...
450686,450687,B003O7ZORU,A3LMRJJPEVZJ3S,Rachel,2,2,5,1297641600,Great kitten food!,I have two boys that love this food! I have be...,great kitten food,3
470392,470393,B000FBQ56M,A1A0MBT5LKK8U9,Lance,2,2,5,1197158400,Awesome butter cookies!!!,These are every bit as good as the other two r...,awesome butter cookies,3
359488,359489,B007M832YY,A3AR4MDJNH4AOL,Dorrie,0,0,5,1319760000,Healthy Indulging,"These cute little circles crunch and taste so,...",healthy indulging,2
532152,532153,B00142B4JE,AFZLZEIQW909R,"Josiejean ""shopping gal""",1,1,5,1265068800,TO DIE FOR,"Wow, their recipe for Seafood Enchilidas is to...",die,3


In [66]:
df_c = df[df['Summary_clean'].isin(score_consistency[score_consistency == 1].index)].copy()

counts = df_c['Summary_clean'].value_counts()
df_c   = df_c[df_c['Summary_clean'].isin(counts[(counts >= 10) & (counts <= 100)].index)].copy()
df_c   = df_c.drop_duplicates(subset=['Text_clean', 'Summary_clean']).copy()

print(f"✅ df_c shape: {df_c.shape}")
print(f"Columns: {df_c.columns.tolist()}")

✅ df_c shape: (12736, 12)
Columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text', 'Summary_clean', 'Text_clean']


In [72]:
df_balanced

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,Summary_clean,summary_word_count
123053,123054,B000EGZ2R6,A3OIM2L0NQH5AT,Dawn,1,1,1,1335312000,Salmon Formula,This smells more like cat food. My dogs ate th...,salmon formula,2
567815,567816,B005K4Q68Q,A3FJ7K6XRP3MPM,J. Voss,1,2,1,1324166400,Worst Hot chocolate ever!,This has to be without a doubt the WORST tasti...,worst hot chocolate ever,4
474056,474057,B000CQC08C,A3OPH5QDQRBNPT,Marilyn49,3,11,1,1250985600,Gluten in Tea!!!,I bought one of the Stash teas-don't remember ...,gluten in tea,3
402692,402693,B003P9XFVO,AED380YNNHAO8,Vicki Quattlebaum,6,10,1,1313712000,Do your research,I have fed BB for 3yrs and right now I have my...,research,3
295705,295706,B003WLC4VC,A33QVK92E9R28T,"Jerry D. Goodwin ""Accountant""",1,1,1,1333152000,Stale Granola Bars,"The Granola bars, with about a month left befo...",stale granola bars,3
...,...,...,...,...,...,...,...,...,...,...,...,...
450686,450687,B003O7ZORU,A3LMRJJPEVZJ3S,Rachel,2,2,5,1297641600,Great kitten food!,I have two boys that love this food! I have be...,great kitten food,3
470392,470393,B000FBQ56M,A1A0MBT5LKK8U9,Lance,2,2,5,1197158400,Awesome butter cookies!!!,These are every bit as good as the other two r...,awesome butter cookies,3
359488,359489,B007M832YY,A3AR4MDJNH4AOL,Dorrie,0,0,5,1319760000,Healthy Indulging,"These cute little circles crunch and taste so,...",healthy indulging,2
532152,532153,B00142B4JE,AFZLZEIQW909R,"Josiejean ""shopping gal""",1,1,5,1265068800,TO DIE FOR,"Wow, their recipe for Seafood Enchilidas is to...",die,3


In [73]:
# ════════════════════════════════════════════════════════
# CELL 3: Model 1 — Text → Summary
# ════════════════════════════════════════════════════════

X1_train, X1_test, y1_train, y1_test = train_test_split(
    df_c['Text_clean'], df_c['Summary_clean'],
    test_size=0.2, random_state=42
)
"""
X1_train, X1_test, y1_train, y1_test = train_test_split(
    df_balanced['Text_clean'], df_balanced['Summary_clean'],
    test_size=0.2, random_state=42
)"""

tfidf_m1    = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X1_train_tf = tfidf_m1.fit_transform(X1_train)

le_summary   = LabelEncoder()
y1_train_enc = le_summary.fit_transform(y1_train)

# Keep only test rows with known labels
mask        = y1_test.isin(le_summary.classes_)
y1_test_enc = le_summary.transform(y1_test[mask])
X1_test_tf  = tfidf_m1.transform(X1_test[mask])

model1 = LogisticRegression(max_iter=500, random_state=42, n_jobs=-1)
model1.fit(X1_train_tf, y1_train_enc)
print(f"✅ Model 1 Accuracy: {accuracy_score(y1_test_enc, model1.predict(X1_test_tf)):.4f}")

✅ Model 1 Accuracy: 0.0925


In [74]:
# ════════════════════════════════════════════════════════
# CELL 4: Model 2 — Summary → Score
# ════════════════════════════════════════════════════════
X2_train, X2_test, y2_train, y2_test = train_test_split(
    df_c['Summary_clean'], df_c['Score'],
    test_size=0.2, random_state=42
)

tfidf_m2    = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X2_train_tf = tfidf_m2.fit_transform(X2_train)
X2_test_tf  = tfidf_m2.transform(X2_test)

model2 = LogisticRegression(max_iter=500, random_state=42, n_jobs=-1)
model2.fit(X2_train_tf, y2_train)
y2_pred = model2.predict(X2_test_tf)
print(f"✅ Model 2 Accuracy: {accuracy_score(y2_test, y2_pred):.4f}")
print(classification_report(y2_test, y2_pred))

✅ Model 2 Accuracy: 0.8807
              precision    recall  f1-score   support

           1       0.75      0.69      0.72       180
           2       0.29      0.08      0.12        63
           3       0.58      0.26      0.35        86
           4       0.60      0.05      0.09       117
           5       0.90      0.99      0.94      2102

    accuracy                           0.88      2548
   macro avg       0.62      0.41      0.45      2548
weighted avg       0.85      0.88      0.85      2548



In [75]:
# run this again
model2.fit(X2_train_tf, y2_train)

# sanity check
print(hasattr(model2, "coef_"))  # should be True

True


In [76]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,Summary_clean,Text_clean
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...,good quality dog food,bought several vitality canned dog food produc...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,advertised,product arrived labeled jumbo salted peanutsth...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...,delight says,confection around centuries light pillowy citr...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...,cough medicine,looking secret ingredient robitussin believe f...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...,great taffy,great taffy great price wide assortment yummy ...


In [79]:
import joblib

joblib.dump(model1,     '/kaggle/working/model1.pkl')
joblib.dump(model2,     '/kaggle/working/model2.pkl')
joblib.dump(tfidf_m1,   '/kaggle/working/tfidf_m1.pkl')
joblib.dump(tfidf_m2,   '/kaggle/working/tfidf_m2.pkl')
joblib.dump(le_summary, '/kaggle/working/le_summary.pkl')
print("✅ Models saved!")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/model1.pkl'

In [81]:
# ════════════════════════════════════════════════════════
# CELL 5: Prediction Function
# ════════════════════════════════════════════════════════
"""import joblib

model1     = joblib.load('/kaggle/working/model1.pkl')
model2     = joblib.load('/kaggle/working/model2.pkl')
tfidf_m1   = joblib.load('/kaggle/working/tfidf_m1.pkl')
tfidf_m2   = joblib.load('/kaggle/working/tfidf_m2.pkl')
le_summary = joblib.load('/kaggle/working/le_summary.pkl')
print("✅ Models loaded!")"""

def predict_full(review_text, customer_score=None):
    cleaned           = clean_text(review_text)
    predicted_summary = le_summary.inverse_transform(
                            model1.predict(tfidf_m1.transform([cleaned])))[0]
    predicted_score   = model2.predict(tfidf_m2.transform([predicted_summary]))[0]
    confidence        = model2.predict_proba(tfidf_m2.transform([predicted_summary]))[0].max() * 100

    print(f"📝 Review  : {review_text[:120]}...")
    print(f"📌 Summary : {predicted_summary}")
    print(f"⭐ Score   : {predicted_score} / 5  ({confidence:.1f}% confidence)")
    if customer_score:
        match = "✅ Match" if customer_score == predicted_score else f"❌ Differs (customer: {customer_score})"
        print(f"👤 Customer: {customer_score} → {match}")
    print()

# ── Test ──
predict_full("This product is absolutely amazing, my dog loves it!", customer_score=5)
predict_full("The product arrived damaged, very disappointing", customer_score=1)
predict_full("The coffee tastes great, will buy again", customer_score=4)

📝 Review  : This product is absolutely amazing, my dog loves it!...
📌 Summary : dogs favorite
⭐ Score   : 5 / 5  (84.0% confidence)
👤 Customer: 5 → ✅ Match

📝 Review  : The product arrived damaged, very disappointing...
📌 Summary : real thing
⭐ Score   : 5 / 5  (81.2% confidence)
👤 Customer: 1 → ❌ Differs (customer: 1)

📝 Review  : The coffee tastes great, will buy again...
📌 Summary : real thing
⭐ Score   : 5 / 5  (81.2% confidence)
👤 Customer: 4 → ❌ Differs (customer: 4)



In [82]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import pandas as pd

# ── Get unique summaries ──
unique_summaries = df_c['Summary_clean'].unique()
print(f"Unique summaries: {len(unique_summaries):,}")

# ── TF-IDF on summaries ──
tfidf_cluster = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
summary_vectors = tfidf_cluster.fit_transform(unique_summaries)

# ── KMeans clustering ──
N_CLUSTERS = 100  # adjust this number
km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
km.fit(summary_vectors)

# ── Map each summary to its cluster ──
cluster_map = dict(zip(unique_summaries, km.labels_))
df_c['summary_cluster'] = df_c['Summary_clean'].map(cluster_map)

# ── For each cluster pick the most frequent summary as the label ──
cluster_labels = (
    df_c.groupby('summary_cluster')['Summary_clean']
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

df_c['Summary_grouped'] = df_c['summary_cluster'].map(cluster_labels)

print(f"\nUnique summaries before: {df_c['Summary_clean'].nunique():,}")
print(f"Unique summaries after : {df_c['Summary_grouped'].nunique():,}")

print(f"\nSample clusters:")
for cluster_id in range(10):
    members = df_c[df_c['summary_cluster'] == cluster_id]['Summary_clean'].unique()
    label   = cluster_labels[cluster_id]
    print(f"  Cluster {cluster_id:>3} → '{label}' : {list(members[:5])}")

Unique summaries: 1,781

Unique summaries before: 1,781
Unique summaries after : 100

Sample clusters:
  Cluster   0 → 'sweet deal' : ['good deal but close expiration date', 'satisfies my sweet tooth in a healthy way', 'sweet memories', 'mostly a good deal', 'sweet deal']
  Cluster   1 → 'amazing tea' : ['amazing chips', 'amazing service', 'excelent tea', 'amazing tea', 'amazing customer service']
  Cluster   2 → 'best tea in the world' : ['pop chips are the best', 'popchips are the best', 'always the best', 'best tea in the world', 'truly the best']
  Cluster   3 → 'love this mix' : ['love this seasoning', 'love this dog food', 'love this treat', 'love this formula', 'love this salt']
  Cluster   4 → 'best quality' : ['top quality', 'high quality tea', 'best quality', 'quality food', 'high quality dog food']
  Cluster   5 → 'deal' : ['best deal ever', 'deal', 'best deal in town', 'best deal', 'best deal around']
  Cluster   6 → 'great for hair' : ['great for dogs', 'great for cooking 

In [83]:

# ── Retrain Model 1 with grouped summaries ──
X1_train, X1_test, y1_train, y1_test = train_test_split(
    df_c['Text_clean'], df_c['Summary_grouped'],  # ✅ use grouped
    test_size=0.2, random_state=42
)

tfidf_m1    = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X1_train_tf = tfidf_m1.fit_transform(X1_train)

le_summary   = LabelEncoder()
y1_train_enc = le_summary.fit_transform(y1_train)

mask        = y1_test.isin(le_summary.classes_)
y1_test_enc = le_summary.transform(y1_test[mask])
X1_test_tf  = tfidf_m1.transform(X1_test[mask])

model1 = LogisticRegression(max_iter=500, random_state=42, solver='saga', n_jobs=1)
model1.fit(X1_train_tf, y1_train_enc)
print(f"✅ Model 1 Accuracy: {accuracy_score(y1_test_enc, model1.predict(X1_test_tf)):.4f}")

print(f"Unique Summary_grouped: {df_c['Summary_grouped'].nunique():,}")
print(f"\nTop 20 grouped summaries:")
print(df_c['Summary_grouped'].value_counts().head(20))


✅ Model 1 Accuracy: 0.2418
Unique Summary_grouped: 100

Top 20 grouped summaries:
Summary_grouped
real thing                   2371
best cereal ever              689
best oatmeal                  606
mm mm good                    407
great tea great price         369
cats favorite                 299
best instant coffee           296
love this mix                 241
heaven                        237
great price great product     226
awesome snack                 216
easy and delicious            214
treat                         212
wonderful stuff               211
excellent pasta               191
amazingly delicious           187
like the taste                185
husband loves                 182
amazing tea                   177
dogs favorite                 173
Name: count, dtype: int64


In [88]:
from sentence_transformers import SentenceTransformer
import numpy as np

# ── Get unique summaries ──
unique_summaries = df_c['Summary_clean'].unique().tolist()
print(f"Unique summaries: {len(unique_summaries):,}")

# ── Generate embeddings ──
model_emb  = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model_emb.encode(unique_summaries, show_progress_bar=True)
print(f"✅ Embeddings shape: {embeddings.shape}")

# ── Add score to embeddings ──
score_map   = df_c.groupby('Summary_clean')['Score'].mean()
scores      = np.array([score_map[s] for s in unique_summaries]).reshape(-1, 1)
scores_norm = scores / 5.0

SEMANTIC_WEIGHT = 0.7
SCORE_WEIGHT    = 0.3
combined = np.hstack([
    embeddings * SEMANTIC_WEIGHT,
    scores_norm * SCORE_WEIGHT
])
print(f"✅ Combined shape: {combined.shape}")

Unique summaries: 2,668


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/84 [00:00<?, ?it/s]

✅ Embeddings shape: (2668, 384)
✅ Combined shape: (2668, 385)


In [89]:
clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.7,
    metric='cosine',
    linkage='average'
)
labels = clustering.fit_predict(combined)

# ✅ Add cluster column to df_c
summary_to_cluster  = dict(zip(unique_summaries, labels))
df_c['cluster']     = df_c['Summary_clean'].map(summary_to_cluster)

print(f"✅ Clusters: {len(set(labels)):,}")
print(f"✅ df_c columns: {df_c.columns.tolist()}")

✅ Clusters: 96
✅ df_c columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text', 'Summary_clean', 'Text_clean', 'cluster']


In [90]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ── Pick most central summary as cluster label ──
summary_to_embedding = dict(zip(unique_summaries, embeddings))
cluster_labels = {}

for cid in set(labels):
    # Get all summaries in this cluster
    members = [unique_summaries[i] for i, l in enumerate(labels) if l == cid]

    if len(members) == 1:
        cluster_labels[cid] = members[0]
        continue

    # Get their embeddings
    member_embeddings = np.array([summary_to_embedding[m] for m in members])

    # Compute centroid
    centroid = member_embeddings.mean(axis=0, keepdims=True)

    # Pick summary closest to centroid
    sims    = cosine_similarity(centroid, member_embeddings)[0]
    best    = members[np.argmax(sims)]
    cluster_labels[cid] = best

df_c['Summary_grouped'] = df_c['cluster'].map(cluster_labels)

print(f"Unique summaries after: {df_c['Summary_grouped'].nunique():,}")
print(f"\nTop 20:")
print(df_c['Summary_grouped'].value_counts().head(20))

Unique summaries after: 96

Top 20:
Summary_grouped
great tasting and healthy        7324
delicious and addictive           789
get any better                    322
calm plus calcium is great        229
wow these are good                183
absolutely horrible               141
best beef jerky on the planet     104
nothing else works                 66
greenies are great                 62
best kcup                          58
beware new formula                 50
strong and bold                    50
worthless                          40
five stars                         38
cannot go without                  33
displeased                         33
stop buying                        29
incorrect information              28
senna leaf                         26
top quality                        26
Name: count, dtype: int64


In [98]:
# !pip install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
import numpy as np

# ── Load lightweight model ──
model_emb = SentenceTransformer('all-MiniLM-L6-v2')

# ── Get unique summaries ──
unique_summaries = df_c['Summary_clean'].unique().tolist()
print(f"Unique summaries: {len(unique_summaries):,}")

# ── Generate embeddings ──
print("⏳ Generating embeddings...")
embeddings = model_emb.encode(unique_summaries, show_progress_bar=True)
print(f"✅ Embeddings shape: {embeddings.shape}")

# ── Agglomerative clustering (groups by actual similarity) ──
print("⏳ Clustering...")
clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=1.2,   # lower = more clusters, higher = fewer clusters
    metric='cosine',
    linkage='average'
)
labels = clustering.fit_predict(embeddings)

print(f"✅ Number of clusters found: {len(set(labels)):,}")

# ── Map each summary to its cluster label ──
summary_to_cluster = dict(zip(unique_summaries, labels))
df_c['cluster'] = df_c['Summary_clean'].map(summary_to_cluster)

# ── Pick most frequent summary in each cluster as the label ──
cluster_labels = (
    df_c.groupby('cluster')['Summary_clean']
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)
df_c['Summary_grouped'] = df_c['cluster'].map(cluster_labels)

print(f"\nUnique summaries before: {df_c['Summary_clean'].nunique():,}")
print(f"Unique summaries after : {df_c['Summary_grouped'].nunique():,}")

# ── Show sample clusters ──
print("\nSample clusters:")
for cid in list(set(labels))[:10]:
    members = [unique_summaries[i] for i, l in enumerate(labels) if l == cid]
    print(f"  '{cluster_labels[cid]}' → {members[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Unique summaries: 2,668
⏳ Generating embeddings...


Batches:   0%|          | 0/84 [00:00<?, ?it/s]

✅ Embeddings shape: (2668, 384)
⏳ Clustering...
✅ Number of clusters found: 1

Unique summaries before: 2,668
Unique summaries after : 1

Sample clusters:
  'best cereal ever' → ['awesome deal', 'warning warning alcohol sugars', 'great marinade', 'best stuff ever', 'tasty chips']


In [92]:
# ── Try different thresholds to find the right one ──
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=threshold,
        metric='cosine',
        linkage='average'
    )
    labels = clustering.fit_predict(embeddings)
    print(f"Threshold {threshold} → {len(set(labels)):,} clusters")

Threshold 0.3 → 1,631 clusters
Threshold 0.4 → 1,147 clusters
Threshold 0.5 → 758 clusters
Threshold 0.6 → 445 clusters
Threshold 0.7 → 213 clusters


In [93]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering

# ── Add score to embeddings so sentiment is considered ──
# Normalize score to same scale as embeddings
score_map     = df_c.groupby('Summary_clean')['Score'].mean()
scores        = np.array([score_map[s] for s in unique_summaries]).reshape(-1, 1)
scores_norm   = scores / 5.0  # normalize to 0-1

# ── Weight: 70% semantic + 30% score ──
SEMANTIC_WEIGHT = 0.7
SCORE_WEIGHT    = 0.3
combined = np.hstack([
    embeddings * SEMANTIC_WEIGHT,
    scores_norm * SCORE_WEIGHT
])

# ── Cluster on combined features ──
clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.5,
    metric='cosine',
    linkage='average'
)
labels = clustering.fit_predict(combined)
print(f"✅ Number of clusters: {len(set(labels)):,}")

# ── Map labels ──
summary_to_cluster = dict(zip(unique_summaries, labels))
df_c['cluster'] = df_c['Summary_clean'].map(summary_to_cluster)

cluster_labels = (
    df_c.groupby('cluster')['Summary_clean']
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)
df_c['Summary_grouped'] = df_c['cluster'].map(cluster_labels)

print(f"Unique summaries before: {df_c['Summary_clean'].nunique():,}")
print(f"Unique summaries after : {df_c['Summary_grouped'].nunique():,}")

# ── Show sample clusters ──
print("\nSample clusters:")
for cid in list(set(labels))[:15]:
    members = [unique_summaries[i] for i, l in enumerate(labels) if l == cid]
    if len(members) > 1:
        print(f"\n  Label : '{cluster_labels[cid]}'")
        print(f"  Members: {members[:6]}")

✅ Number of clusters: 555
Unique summaries before: 2,668
Unique summaries after : 555

Sample clusters:

  Label : 'try with lemon juice'
  Members: ['try with lemon juice', 'mostly lime', 'limey', 'lime', 'lime flavored spicey nut', 'lime lovers']

  Label : 'great price great product'
  Members: ['awesome deal', 'amazing service', 'service was good', 'great product great customer service', 'another great product', 'great product great service']

  Label : 'amazing tea'
  Members: ['best decaf ever', 'tea should get the nobel peace prize', 'good tea wish it came in loose tea', 'worst tea i have ever tried', 'excelent tea', 'amazing tea']

  Label : 'deelicious'
  Members: ['deelicious', 'delectable', 'deefreakinglicious', 'delisious']

  Label : 'daughters first choice'
  Members: ['daughters first choice', 'dad would approve']

  Label : 'best coffee in the world'
  Members: ['smooth coffee highly recommended', 'good bold coffee', 'greatest coffee', 'worst coffee ever', 'best coffee 

In [94]:
# ── Re-cluster with higher threshold to get fewer groups ──
clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.8,  # ✅ higher = fewer clusters
    metric='cosine',
    linkage='average'
)
labels = clustering.fit_predict(combined)
print(f"✅ Number of clusters: {len(set(labels)):,}")

# ── Remap ──
summary_to_cluster = dict(zip(unique_summaries, labels))
df_c['cluster']         = df_c['Summary_clean'].map(summary_to_cluster)
cluster_labels          = (
    df_c.groupby('cluster')['Summary_clean']
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)
df_c['Summary_grouped'] = df_c['cluster'].map(cluster_labels)

print(f"Unique summaries before: {df_c['Summary_clean'].nunique():,}")
print(f"Unique summaries after : {df_c['Summary_grouped'].nunique():,}")
print(f"\nTop 20:")
print(df_c['Summary_grouped'].value_counts().head(20))

✅ Number of clusters: 22
Unique summaries before: 2,668
Unique summaries after : 22

Top 20:
Summary_grouped
best cereal ever                               9584
worthless                                       225
rancid                                           53
excellant                                        50
cannot tell the difference                       43
waste your time                                  16
irresistable                                     15
offensive gas producer                            9
jury is still                                     8
bal                                               8
doomhammer approved                               7
none of my cats will touch this stuff             7
expired stock                                     6
go for pinhead gunpowder instead                  4
gerbers fruits are cooked                         4
fluoride content an issue                         3
warning contains menadione                        2
stops f

In [95]:
for threshold in [0.55, 0.60, 0.65, 0.70]:
    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=threshold,
        metric='cosine',
        linkage='average'
    )
    labels = clustering.fit_predict(combined)
    print(f"Threshold {threshold} → {len(set(labels)):,} clusters")

Threshold 0.55 → 407 clusters
Threshold 0.6 → 267 clusters
Threshold 0.65 → 168 clusters
Threshold 0.7 → 96 clusters


In [99]:
# ── Re-cluster with threshold 0.6 ──
clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.6,
    metric='cosine',
    linkage='average'
)
labels = clustering.fit_predict(combined)
print(f"✅ Number of clusters: {len(set(labels)):,}")

# ── Remap ──
summary_to_cluster      = dict(zip(unique_summaries, labels))
df_c['cluster']         = df_c['Summary_clean'].map(summary_to_cluster)
cluster_labels          = (
    df_c.groupby('cluster')['Summary_clean']
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)
df_c['Summary_grouped'] = df_c['cluster'].map(cluster_labels)

print(f"Unique summaries after: {df_c['Summary_grouped'].nunique():,}")
print(f"\nTop 20:")
print(df_c['Summary_grouped'].value_counts().head(20))

✅ Number of clusters: 267
Unique summaries after: 267

Top 20:
Summary_grouped
easy and delicious         2056
amazing tea                1210
best dog food               877
best on the market          455
perfection                  441
wonderful stuff             408
best cookies ever           343
best cereal ever            315
great peanut butter         284
love it love it love        185
best popcorn ever           139
finally found               133
heaven                      122
delicious and addictive     106
nothing better              102
best seasoning               98
love this salt               89
yay                          79
terrible product             76
deelicious                   73
Name: count, dtype: int64


In [100]:
# ── Re-cluster with threshold 0.7 ──
clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.7,
    metric='cosine',
    linkage='average'
)
labels = clustering.fit_predict(combined)
print(f"✅ Number of clusters: {len(set(labels)):,}")

# ── Remap ──
summary_to_cluster      = dict(zip(unique_summaries, labels))
df_c['cluster']         = df_c['Summary_clean'].map(summary_to_cluster)
cluster_labels          = (
    df_c.groupby('cluster')['Summary_clean']
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)
df_c['Summary_grouped'] = df_c['cluster'].map(cluster_labels)

print(f"Unique summaries after: {df_c['Summary_grouped'].nunique():,}")
print(f"\nTop 20:")
print(df_c['Summary_grouped'].value_counts().head(20))

✅ Number of clusters: 96
Unique summaries after: 96

Top 20:
Summary_grouped
best cereal ever              7324
deelicious                     789
die                            322
great for hair                 229
finally found                  183
absolutely disgusting          141
staple                         104
got to try                      66
best brownies ever              62
best kcup                       58
excellant                       50
bold and beautiful              50
worthless                       40
five stars                      38
cannot do without               33
horrid                          33
never order                     29
cannot tell the difference      28
beautiful tree                  26
top of the line                 26
Name: count, dtype: int64


In [101]:
# ── Top 20 most frequent summaries ──
top20 = df_c['Summary_clean'].value_counts().index.tolist()
print("Top 20 summaries:")
print(top20)

# ── Map everything else to 'other' ──
df_c['Summary_grouped'] = df_c['Summary_clean'].apply(
    lambda x: x if x in top20 else 'other'
)

print(f"\nUnique summaries: {df_c['Summary_grouped'].nunique():,}")
print(f"\nDistribution:")
print(df_c['Summary_grouped'].value_counts())

Top 20 summaries:
['best cereal ever', 'deelicious', 'best popcorn ever', 'perfection', 'finally found', 'amazing tea', 'heaven', 'best cookies ever', 'love this sauce', 'easy and delicious', 'wonderful stuff', 'best dog food', 'best gum ever', 'die', 'amazing stuff', 'love it love it love', 'best hot sauce ever', 'absolute best', 'best on the market', 'awesome snack', 'great hot chocolate', 'great price great product', 'love at first bite', 'best sauce ever', 'nothing better', 'best seasoning', 'best tea in the world', 'amazingly delicious', 'always good', 'absolutely love', 'bomb', 'best cookie ever', 'best tea i have ever', 'worthless', 'super product', 'best stuff ever', 'yay', 'happy to find', 'great product great service', 'delicious and good', 'absolutely amazing', 'unbelievably good', 'best decaf ever', 'got to try', 'delicious and refreshing', 'great way to start the day', 'get any better', 'rancid', 'absolutely disgusting', 'awesome deal', 'everyone loves', 'best crackers eve

In [103]:
# ── Balance Summary_grouped — max 200 rows per group ──
df_train = df_c.groupby('Summary_grouped', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 200), random_state=42)
).reset_index(drop=True)

print(f"✅ Balanced shape: {df_train.shape}")
print(f"Unique summaries: {df_train['Summary_grouped'].nunique():,}")
print(f"\nDistribution:")
print(df_train['Summary_grouped'].value_counts())

✅ Balanced shape: (10051, 14)
Unique summaries: 2,668

Distribution:
Summary_grouped
best cereal ever                                                68
deelicious                                                      57
best popcorn ever                                               54
perfection                                                      53
finally found                                                   50
                                                                ..
addict great for low carb                                        1
add honey                                                        1
acceptable but not fantastic                                     1
absotively posilutely delicious                                  1
absolutely the yummiest of the blue diamond flavored almonds     1
Name: count, Length: 2668, dtype: int64


/tmp/ipykernel_7868/2706536292.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_train = df_c.groupby('Summary_grouped', group_keys=False).apply(


In [105]:
X1_train, X1_test, y1_train, y1_test = train_test_split(
    df_train['Text_clean'], df_train['Summary_grouped'],
    test_size=0.2, random_state=42
)

tfidf_m1    = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X1_train_tf = tfidf_m1.fit_transform(X1_train)

le_summary   = LabelEncoder()
y1_train_enc = le_summary.fit_transform(y1_train)

mask        = y1_test.isin(le_summary.classes_)
y1_test_enc = le_summary.transform(y1_test[mask])
X1_test_tf  = tfidf_m1.transform(X1_test[mask])

model1 = LogisticRegression(max_iter=500, random_state=42, solver='saga', n_jobs=1)
model1.fit(X1_train_tf, y1_train_enc)
print(f"✅ Model 1 Accuracy: {accuracy_score(y1_test_enc, model1.predict(X1_test_tf)):.4f}")

✅ Model 1 Accuracy: 0.0882


In [106]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    df_train['Summary_grouped'], df_train['Score'],
    test_size=0.2, random_state=42
)

tfidf_m2    = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X2_train_tf = tfidf_m2.fit_transform(X2_train)
X2_test_tf  = tfidf_m2.transform(X2_test)

model2 = LogisticRegression(max_iter=500, random_state=42, solver='saga', n_jobs=1)
model2.fit(X2_train_tf, y2_train)
print(f"✅ Model 2 Accuracy: {accuracy_score(y2_test, model2.predict(X2_test_tf)):.4f}")

✅ Model 2 Accuracy: 0.9224


In [107]:
# Test with long reviews
long_reviews = [
    # Positive review
    """I have been using this dog food for over a year now and my dogs absolutely love it.
    The quality is outstanding, the ingredients are natural and healthy, and the price is
    very reasonable. My vet also recommended it and said my dogs look healthier than ever.
    Will definitely keep buying this product.""",

    # Negative review
    """This product was a complete disaster. It arrived damaged and the packaging was torn.
    The smell was terrible and my cat refused to eat it. I contacted customer service but
    got no response. Total waste of money and I will never buy this again.""",

    # Neutral review
    """The coffee is okay, nothing special. It tastes like any regular instant coffee you
    can find at the store. The price is a bit high for what you get but the delivery was
    fast and the packaging was intact. Might buy again if on sale.""",
]

print("=" * 60)
for review in long_reviews:
    predict_full(review)
    print("=" * 60)

📝 Review  : I have been using this dog food for over a year now and my dogs absolutely love it. 
    The quality is outstanding, the...
📌 Summary : best dog food
⭐ Score   : 5 / 5  (99.0% confidence)

📝 Review  : This product was a complete disaster. It arrived damaged and the packaging was torn. 
    The smell was terrible and my ...
📌 Summary : best cereal ever
⭐ Score   : 5 / 5  (99.1% confidence)

📝 Review  : The coffee is okay, nothing special. It tastes like any regular instant coffee you 
    can find at the store. The price...
📌 Summary : best cereal ever
⭐ Score   : 5 / 5  (99.1% confidence)



In [110]:
import re
import pandas as pd
import os
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import nltk
nltk.download('stopwords', quiet=True)

STOP_WORDS = set(stopwords.words('english'))

# ════════════════════════════════════════════════════════
# CELL 1: Clean Text
# ════════════════════════════════════════════════════════
def collapse_repeated_chars(text):
    return re.sub(r'(.)\1{2,}', r'\1\1', str(text))

def strip_leading_trailing_stopwords(text):
    words = text.split()
    while words and words[0] in STOP_WORDS:
        words.pop(0)
    while words and words[-1] in STOP_WORDS:
        words.pop(-1)
    return ' '.join(words)

def clean_text(text):
    text = str(text).lower()
    text = collapse_repeated_chars(text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(w for w in text.split() if w not in STOP_WORDS)

def clean_summary(text):
    text = str(text).lower()
    text = collapse_repeated_chars(text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return strip_leading_trailing_stopwords(text)

df['Text_clean']    = df['Text'].apply(clean_text)
df['Summary_clean'] = df['Summary'].apply(clean_summary)
print(f"✅ Cleaning done: {df.shape}")

# ════════════════════════════════════════════════════════
# CELL 2: Filter & Balance
# ════════════════════════════════════════════════════════
score_consistency = df.groupby('Summary_clean')['Score'].nunique()
df_c = df[df['Summary_clean'].isin(score_consistency[score_consistency == 1].index)].copy()

counts = df_c['Summary_clean'].value_counts()
df_c   = df_c[df_c['Summary_clean'].isin(counts[(counts >= 10) & (counts <= 100)].index)].copy()
df_c   = df_c.drop_duplicates(subset=['Text_clean', 'Summary_clean']).copy()

# Balance — max 2000 per score
df_c = df_c.groupby('Score', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 2000), random_state=42)
).reset_index(drop=True)

print(f"✅ df_c shape: {df_c.shape}")
print(df_c['Score'].value_counts().sort_index())

# ════════════════════════════════════════════════════════
# CELL 3: Train Model — Text → Score directly
# ════════════════════════════════════════════════════════
X_train, X_test, y_train, y_test = train_test_split(
    df_c['Text_clean'], df_c['Score'],
    test_size=0.2, random_state=42, stratify=df_c['Score']
)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf  = tfidf.transform(X_test)

model = LogisticRegression(max_iter=500, random_state=42, solver='saga', n_jobs=1)
model.fit(X_train_tf, y_train)
y_pred = model.predict(X_test_tf)

print(f"✅ Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

# ════════════════════════════════════════════════════════
# CELL 4: Save Model
# ════════════════════════════════════════════════════════
# ── Save to current directory ──
save_dir = os.getcwd()
joblib.dump(model, os.path.join(save_dir, 'model.pkl'))
joblib.dump(tfidf, os.path.join(save_dir, 'tfidf.pkl'))
print(f"✅ Model saved to: {save_dir}")

# ════════════════════════════════════════════════════════
# CELL 5: Prediction Function
# ════════════════════════════════════════════════════════
def predict_review(review_text, customer_score=None):
    cleaned         = clean_text(review_text)
    features        = tfidf.transform([cleaned])
    predicted_score = model.predict(features)[0]
    confidence      = model.predict_proba(features)[0].max() * 100

    # Generate simple summary based on score
    summary_map = {
        1: 'very disappointing',
        2: 'below expectations',
        3: 'average product',
        4: 'good product',
        5: 'excellent product'
    }
    predicted_summary = summary_map[predicted_score]

    print(f"📝 Review  : {review_text[:120]}...")
    print(f"📌 Summary : {predicted_summary}")
    print(f"⭐ Score   : {predicted_score} / 5  ({confidence:.1f}% confidence)")
    if customer_score:
        match = "✅ Match" if customer_score == predicted_score else f"❌ Differs (customer: {customer_score})"
        print(f"👤 Customer: {customer_score} → {match}")
    print()

# ── Test ──
predict_review("I have been using this dog food for over a year, my dogs absolutely love it, outstanding quality!", customer_score=5)
predict_review("This product was a complete disaster, arrived damaged, terrible smell, total waste of money", customer_score=1)
predict_review("The coffee is okay, nothing special, tastes like regular instant coffee, price is a bit high", customer_score=3)

✅ Cleaning done: (568454, 12)


/tmp/ipykernel_7868/3142547464.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_c = df_c.groupby('Score', group_keys=False).apply(


✅ df_c shape: (3191, 12)
Score
1     567
2     103
3     197
4     324
5    2000
Name: count, dtype: int64
✅ Accuracy: 0.7527
              precision    recall  f1-score   support

           1       0.84      0.61      0.70       114
           2       0.00      0.00      0.00        21
           3       1.00      0.05      0.10        39
           4       0.59      0.20      0.30        65
           5       0.74      0.99      0.85       400

    accuracy                           0.75       639
   macro avg       0.64      0.37      0.39       639
weighted avg       0.74      0.75      0.69       639

✅ Model saved to: /content
📝 Review  : I have been using this dog food for over a year, my dogs absolutely love it, outstanding quality!...
📌 Summary : excellent product
⭐ Score   : 5 / 5  (91.9% confidence)
👤 Customer: 5 → ✅ Match

📝 Review  : This product was a complete disaster, arrived damaged, terrible smell, total waste of money...
📌 Summary : very disappointing
⭐ Score   : 1 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [111]:
# ── Better balance — equal rows per score ──
min_count = df_c['Score'].value_counts().min()
print(f"Min count per score: {min_count}")

df_c = df_c.groupby('Score', group_keys=False).apply(
    lambda x: x.sample(min(len(x), min_count), random_state=42)
).reset_index(drop=True)

print(f"✅ Balanced shape: {df_c.shape}")
print(df_c['Score'].value_counts().sort_index())

Min count per score: 103
✅ Balanced shape: (515, 12)
Score
1    103
2    103
3    103
4    103
5    103
Name: count, dtype: int64


/tmp/ipykernel_7868/4283505471.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_c = df_c.groupby('Score', group_keys=False).apply(


In [113]:
!pip install transformers torch

In [115]:
from transformers import pipeline

print("⏳ Loading model...")
summarizer = pipeline("text-generation", model="gpt2")
print("✅ Model loaded!")

def summarize_review():
    print("\n" + "="*60)
    topic      = input("📌 Topic: ").strip()
    print(f"\n✍️  Write your review about '{topic}' (Enter twice to finish):\n")
    lines = []
    while True:
        line = input()
        if line == "":
            break
        lines.append(line)
    review_text = " ".join(lines)

    score = int(input("\n⭐ Score (1-5): ").strip())

    # ── Summarize using prompt ──
    prompt    = f"Summarize this review in one sentence: {review_text}\nSummary:"
    result    = summarizer(prompt, max_new_tokens=50, do_sample=False)[0]['generated_text']
    summary   = result.split("Summary:")[-1].strip().split("\n")[0]

    print(f"\n📌 Topic   : {topic}")
    print(f"📋 Summary : {summary}")
    print(f"⭐ Score   : {score} / 5")

# ── Run ──
summarize_review()

⏳ Loading model...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Model loaded!

📌 Topic: science

✍️  Write your review about 'science' (Enter twice to finish):

.


⭐ Score (1-5): 


ValueError: invalid literal for int() with base 10: ''